# Run an election from a hand-built `data` dictionary

Minimal path, for a collaborator:

1. build the `data` dictionary by hand (preferences + candidates),
2. pick the **preference-based** response function,
3. run **N elections** under plurality.

No perception / HGF / inference here, and **no beliefs**: we supply
preferences and candidates directly and aggregate votes.

## 1. The `data` dictionary

Response functions and voting rules consume a single dict. For the
**preference-based** response function we only need two groups:

```python
data = {
    "preferences": {"mean": ..., "precision": ...},   # (n_agents, n_pref)
    "candidates":  {"mean": ..., "precision": ...},   # (n_candidates, n_pref)
}
```

- **preferences** — each agent's ideal point (`mean`) and how sharply it holds it (`precision`).
- **candidates** — each candidate's position (`mean`) and sharpness (`precision`).

There is **no `beliefs` key** here: beliefs come from the HGF perception
module, which we are not using yet, and `response_function_pref` does not read
them. (A belief-aware response function such as `response_function` *would*
require an extra `"beliefs": {"mean", "precision"}` group.)

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from eci.decision import response_function_pref
from eci.voting import _vote_plurality

key = jax.random.PRNGKey(0)

In [ ]:
n_agents, n_candidates, n_pref = 200, 4, 1

# --- preferences: ideal points ~ N(0, 1.5), held with precision 1.0 ---
key, k = jax.random.split(key)
pref_mean = jax.random.normal(k, (n_agents, n_pref)) * 1.5
pref_precision = jnp.full((n_agents, n_pref), 1.0)

# --- candidates: fixed positions, extreme (C0, C3) to central (C1, C2) ---
cand_mean = jnp.array([[-3.0], [-1.0], [1.0], [3.0]])  # (n_candidates, n_pref)
cand_precision = jnp.full((n_candidates, n_pref), 5.0)

data = {
    "preferences": {"mean": pref_mean, "precision": pref_precision},
    "candidates": {"mean": cand_mean, "precision": cand_precision},
}

# sanity check on shapes
for group, d in data.items():
    print(group, {k: tuple(v.shape) for k, v in d.items()})

## 2. The preference-based response function

`response_function_pref` scores each candidate by **−KL(candidate ‖ preference)**:
an agent favours candidates whose policy distribution is closest to its own
preference, then samples a vote from a softmax over those scores. It uses only
`preferences` and `candidates`.

## 3. A single election

`_vote_plurality(data, response_function, key)` returns a result dict:
`winner`, `votes_per_candidate`, `votes_matrix`, `softmax`, `candidate_utilities`.

In [ ]:
key, k = jax.random.split(key)
result = _vote_plurality(data, response_function_pref, k)

print("winner:", int(result["winner"]))
print("votes per candidate:", np.asarray(result["votes_per_candidate"]))

## 4. Run N elections

An election is stochastic (votes are sampled), so we repeat it over fresh PRNG
keys and look at how often each candidate wins. The loop below is exactly what
`Environment.run_n_simulation` does internally — written out here to stay free
of the perception/HGF machinery.

In [ ]:
def run_n_elections(data, response_function, key, n):
    """Run n independent plurality elections; mirrors run_n_simulation."""
    results = {}
    for i in range(n):
        key, subkey = jax.random.split(key)
        results[i] = _vote_plurality(data, response_function, subkey)
    return results


key, k = jax.random.split(key)
results = run_n_elections(data, response_function_pref, k, n=200)

winners = np.array([int(results[i]["winner"]) for i in results])
win_freq = np.bincount(winners, minlength=n_candidates) / len(winners)

for c, f in enumerate(win_freq):
    print(f"C{c}: win frequency = {f:.3f}")

Central candidates (C1, C2) should win most often: most agents' ideal points
sit near 0, so the central candidates minimise KL(candidate ‖ preference).

## Notes & next steps

- **Built-in equivalent:** `env.run_n_simulation(_vote_plurality, data, response_function_pref, key, n)` does the same loop, but requires constructing an `Environment` (which builds the HGF model we are skipping here).
- **Quadratic voting:** swap `_vote_plurality` for `eci.voting._vote_quadratic` — same `data`, same response function.
- **Cleaner aggregation:** `eci.metrics.winner_frequencies(winners, n_candidates)` returns win frequencies with a standard error.
- **Adding beliefs later:** once the HGF perception module is wired in, add a `"beliefs": {"mean", "precision"}` group to `data` and switch to a belief-aware response function (e.g. `response_function`).